<a href="https://colab.research.google.com/github/quirocode/Medical-data-privacy-attack/blob/main/notebooks/linkage_attack_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. Generamos nuestra propia Base de Datos Médica Sintética (Dm)
# Nota que NO hay nombres aquí, supuestamente es "segura".
hospital_data = pd.DataFrame({
    'ID_Paciente': ['P001', 'P002', 'P003', 'P004'],
    'Edad': [45, 85, 22, 45],
    'Genero': ['M', 'H', 'M', 'H'],
    'Cod_Postal': ['15011', '15034', '15011', '15046'],
    'Diagnostico': ['Diabetes', 'Hipertension', 'Asma', 'Migraña']
})

# 2. Generamos el Padrón Público Sintético (Dp)
# Esta base sí tiene nombres reales y es de acceso público.
public_registry = pd.DataFrame({
    'Nombre_Real': ['Ana Lopez', 'Carlos Perez', 'Maria Salas', 'Juan Ruiz', 'Luis Silva'],
    'Edad': [45, 85, 22, 45, 30],
    'Genero': ['M', 'H', 'M', 'H', 'H'],
    'Cod_Postal': ['15011', '15034', '15011', '15046', '15011']
})

# 3. Ejecutamos el Ataque de Enlace (Linkage Attack)
# Le decimos a Python que cruce ambas tablas usando nuestros Cuasi-Identificadores
ataque_exitoso = pd.merge(hospital_data, public_registry,
                          on=['Edad', 'Genero', 'Cod_Postal'],
                          how='inner')

# 4. Mostramos los Resultados
print("¡Ataque completado! Pacientes re-identificados:")
print(ataque_exitoso[['Nombre_Real', 'Diagnostico', 'Edad', 'Cod_Postal']])

In [ ]:
!pip install faker
import pandas as pd
from faker import Faker
import random

# 1. Configuración del Entorno Sintético
# Usamos el locale 'es_ES' o 'es_MX' para generar nombres en español
fake = Faker('es_ES')

NUM_REGISTROS = 10000
# Definimos una lista de diagnósticos y limitamos los códigos postales para forzar coincidencias
ENFERMEDADES = ['Diabetes', 'Hipertensión', 'Asma', 'Migraña', 'Ansiedad', 'VIH', 'Ninguna']
CODIGOS_POSTALES = [str(fake.random_int(min=15000, max=15050)) for _ in range(50)]

print(f"Generando {NUM_REGISTROS} registros sintéticos con Faker... Esto puede tomar unos segundos.")

# 2. Generación del "Universo" de Datos
poblacion = []
for i in range(NUM_REGISTROS):
    poblacion.append({
        'Nombre_Real': fake.name(),
        'Edad': random.randint(18, 90),
        'Genero': random.choice(['M', 'F']),
        'Cod_Postal': random.choice(CODIGOS_POSTALES),
        'Diagnostico': random.choice(ENFERMEDADES)
    })

df_poblacion = pd.DataFrame(poblacion)

# 3. Separación de las Bases de Datos
# Dp: Padrón Público (Tiene nombres y QIs, pero NO diagnósticos)
public_registry = df_poblacion[['Nombre_Real', 'Edad', 'Genero', 'Cod_Postal']].copy()

# Dm: Base del Hospital (Tiene diagnósticos y QIs, supuestamente anonimizada sin nombres)
hospital_data = df_poblacion[['Edad', 'Genero', 'Cod_Postal', 'Diagnostico']].copy()
# Le asignamos un ID falso de hospital
hospital_data['ID_Paciente'] = ['P' + str(i).zfill(5) for i in range(NUM_REGISTROS)]
# Desordenamos la base del hospital para que no coincida fila por fila con el padrón
hospital_data = hospital_data.sample(frac=1).reset_index(drop=True)

# 4. LA METODOLOGÍA: El Ataque de Enlace (Linkage Attack)
print("Ejecutando el algoritmo de re-identificación...")

# Definimos nuestros Cuasi-Identificadores (QI)
qis = ['Edad', 'Genero', 'Cod_Postal']

# Para estar 100% seguros de a QUIÉN re-identificamos, filtramos solo los individuos
# que son ÚNICOS en ambas bases de datos basándonos en sus QIs.
public_unicos = public_registry.drop_duplicates(subset=qis, keep=False)
hospital_unicos = hospital_data.drop_duplicates(subset=qis, keep=False)

# Ejecutamos el cruce (Merge)
ataque_exitoso = pd.merge(hospital_unicos, public_unicos, on=qis, how='inner')

# 5. LOS RESULTADOS
tasa_exito = (len(ataque_exitoso) / NUM_REGISTROS) * 100

print(f"\n" + "="*40)
print(f"🔬 RESULTADOS DEL EXPERIMENTO CIIS 2026")
print(f"="*40)
print(f"Población total analizada: {NUM_REGISTROS} pacientes.")
print(f"Pacientes re-identificados con éxito: {len(ataque_exitoso)}")
print(f"Tasa de éxito del ataque: {tasa_exito:.2f}%")
print(f"-"*40)
print("Ejemplo de registros vulnerados (Secreto Médico Expuesto):")
print(ataque_exitoso[['Nombre_Real', 'Diagnostico', 'Edad', 'Cod_Postal']].head())

In [ ]:
import os

# 1. Tu nuevo token de acceso directo de Kaggle
# (En investigaciones futuras, recuerda mantener este código en secreto)
API_TOKEN = "KGAT_950ef98680cbc7df25e0d5ae167e06a7"

# 2. Creamos la carpeta oculta requerida por Kaggle
os.makedirs('/root/.kaggle', exist_ok=True)

# 3. Guardamos el token en el archivo 'access_token'
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(API_TOKEN)

# 4. Asignamos permisos de seguridad estrictos (lectura/escritura solo para ti)
os.chmod('/root/.kaggle/access_token', 0o600)

print("¡Nuevo Token de Kaggle configurado exitosamente!")

In [ ]:
# Descargamos el dataset masivo directamente a la nube
!kaggle datasets download -d drscarlat/syntheacovid100k

# Descomprimimos el archivo descargado
!unzip -q syntheacovid100k.zip -d synthea_data

print("¡Dataset masivo descargado y listo para el análisis!")

In [ ]:
import os
import pandas as pd

print("Buscando las rutas exactas de los archivos masivos...")

ruta_pacientes = ""
ruta_condiciones = ""

# 1. Rastreador automático en el entorno de Colab
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.lower() == 'patients.csv':
            ruta_pacientes = os.path.join(root, file)
        elif file.lower() == 'conditions.csv':
            ruta_condiciones = os.path.join(root, file)

# Verificamos si se encontraron los archivos
if not ruta_pacientes or not ruta_condiciones:
    print("❌ Error: No se encontraron los archivos CSV. Asegúrate de que la celda de '!unzip' terminó de ejecutarse correctamente.")
else:
    print(f"✅ Archivos encontrados exitosamente:\n- {ruta_pacientes}\n- {ruta_condiciones}")

    # 2. Cargar los datos desde las rutas dinámicas
    print("\nCargando bases de datos masivas... (Esto tomará un momento)")
    df_patients = pd.read_csv(ruta_pacientes)
    df_conditions = pd.read_csv(ruta_condiciones)

    # 3. Preparación y Limpieza de Datos (Data Wrangling)
    print("Procesando y uniendo historiales clínicos...")

    # En Synthea, la columna 'Id' de pacientes equivale a 'PATIENT' en condiciones
    df_patients = df_patients.rename(columns={'Id': 'PATIENT'})

    df_patients = df_patients[['PATIENT', 'FIRST', 'LAST', 'BIRTHDATE', 'GENDER', 'ZIP']]
    df_conditions = df_conditions[['PATIENT', 'DESCRIPTION']]

    df_master = pd.merge(df_patients, df_conditions, on='PATIENT', how='inner')

    # Eliminamos registros sin código postal para mantener la integridad de los Cuasi-Identificadores
    df_master = df_master.dropna(subset=['ZIP'])
    df_master['Nombre_Real'] = df_master['FIRST'].astype(str) + ' ' + df_master['LAST'].astype(str)

    print(f"Base maestra consolidada: {len(df_master)} registros médicos encontrados.")

    # 4. Construcción del Escenario de Ataque
    qis = ['BIRTHDATE', 'GENDER', 'ZIP']
    public_registry = df_master[['Nombre_Real'] + qis].copy().drop_duplicates()

    hospital_data = df_master[qis + ['DESCRIPTION']].copy()
    hospital_data = hospital_data.sample(frac=1).reset_index(drop=True)

    # 5. LA METODOLOGÍA: Linkage Attack sobre Big Data
    print("Ejecutando algoritmo de re-identificación en el ecosistema...")

    public_unicos = public_registry.drop_duplicates(subset=qis, keep=False)
    hospital_unicos = hospital_data.drop_duplicates(subset=qis, keep=False)

    ataque_exitoso = pd.merge(hospital_unicos, public_unicos, on=qis, how='inner')

    # 6. LOS RESULTADOS
    tasa_exito = (len(ataque_exitoso) / len(df_master)) * 100

    print("\n" + "="*60)
    print("🔬 RESULTADOS DEL EXPERIMENTO (BIG DATA) - CIIS 2026")
    print("="*60)
    print(f"Población clínica total analizada:  {len(df_master)} registros.")
    print(f"Historiales médicos vulnerados:     {len(ataque_exitoso)}")
    print(f"Tasa de éxito de re-identificación: {tasa_exito:.2f}%")
    print("-" * 60)
    print("Primeros 5 registros expuestos (Nombre y Enfermedad):")
    print(ataque_exitoso[['Nombre_Real', 'DESCRIPTION', 'BIRTHDATE', 'ZIP']].head())

In [ ]:
import os
import pandas as pd

print("Buscando las rutas exactas de los archivos masivos...")

ruta_pacientes = ""
ruta_condiciones = ""

# Rastreador automático en el entorno de Colab
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.lower() == 'patients.csv':
            ruta_pacientes = os.path.join(root, file)
        elif file.lower() == 'conditions.csv':
            ruta_condiciones = os.path.join(root, file)

if not ruta_pacientes or not ruta_condiciones:
    print("❌ Error: No se encontraron los archivos CSV. Ejecuta primero la celda de descarga.")
else:
    print("✅ Archivos encontrados. Cargando bases de datos masivas...")
    df_patients = pd.read_csv(ruta_pacientes)
    df_conditions = pd.read_csv(ruta_condiciones)

    print("Procesando historiales clínicos con NUEVAS variables demográficas...")

    df_patients = df_patients.rename(columns={'Id': 'PATIENT'})

    # AQUÍ ESTÁ EL CAMBIO CLAVE: Agregamos RACE y MARITAL a la extracción
    df_patients = df_patients[['PATIENT', 'FIRST', 'LAST', 'BIRTHDATE', 'GENDER', 'ZIP', 'RACE', 'MARITAL']]
    df_conditions = df_conditions[['PATIENT', 'DESCRIPTION']]

    df_master = pd.merge(df_patients, df_conditions, on='PATIENT', how='inner')

    # Limpieza: Eliminamos registros que no tengan estas nuevas variables
    df_master = df_master.dropna(subset=['ZIP', 'RACE', 'MARITAL'])
    df_master['Nombre_Real'] = df_master['FIRST'].astype(str) + ' ' + df_master['LAST'].astype(str)

    print(f"Base maestra consolidada: {len(df_master)} registros médicos aptos para el ataque.")

    # 4. Construcción del Escenario de Ataque con el nuevo arsenal
    # Agregamos las nuevas dimensiones a nuestros Cuasi-Identificadores (QIs)
    qis = ['BIRTHDATE', 'GENDER', 'ZIP', 'RACE', 'MARITAL']

    public_registry = df_master[['Nombre_Real'] + qis].copy().drop_duplicates()
    hospital_data = df_master[qis + ['DESCRIPTION']].copy()
    hospital_data = hospital_data.sample(frac=1).reset_index(drop=True)

    # 5. LA METODOLOGÍA: Ejecutando el cruce dimensional
    print("Ejecutando algoritmo letal de re-identificación...")

    public_unicos = public_registry.drop_duplicates(subset=qis, keep=False)
    hospital_unicos = hospital_data.drop_duplicates(subset=qis, keep=False)

    ataque_exitoso = pd.merge(hospital_unicos, public_unicos, on=qis, how='inner')

    # 6. LOS RESULTADOS
    tasa_exito = (len(ataque_exitoso) / len(df_master)) * 100

    print("\n" + "="*65)
    print("🔬 RESULTADOS DEL EXPERIMENTO LETAL (BIG DATA) - CIIS 2026")
    print("="*65)
    print(f"Población clínica total analizada:  {len(df_master)} registros.")
    print(f"Historiales médicos vulnerados:     {len(ataque_exitoso)}")
    print(f"Tasa de éxito de re-identificación: {tasa_exito:.2f}%")
    print("-" * 65)
    print("Primeros 5 registros expuestos (Nombre, Enfermedad, Raza y Estado Civil):")
    # Mostramos las nuevas variables en la tabla final para probar el impacto
    print(ataque_exitoso[['Nombre_Real', 'DESCRIPTION', 'RACE', 'MARITAL', 'ZIP']].head())

In [ ]:
import os
import pandas as pd

print("Buscando las rutas exactas de los archivos masivos...")

ruta_pacientes = ""
ruta_condiciones = ""

for root, dirs, files in os.walk('.'):
    for file in files:
        if file.lower() == 'patients.csv':
            ruta_pacientes = os.path.join(root, file)
        elif file.lower() == 'conditions.csv':
            ruta_condiciones = os.path.join(root, file)

if not ruta_pacientes or not ruta_condiciones:
    print("❌ Error: No se encontraron los archivos CSV.")
else:
    print("✅ Archivos encontrados. Cargando bases de datos masivas...")
    df_patients = pd.read_csv(ruta_pacientes)
    df_conditions = pd.read_csv(ruta_condiciones)

    print("Procesando historiales clínicos con Imputación de Datos...")

    df_patients = df_patients.rename(columns={'Id': 'PATIENT'})
    df_patients = df_patients[['PATIENT', 'FIRST', 'LAST', 'BIRTHDATE', 'GENDER', 'ZIP', 'RACE', 'MARITAL']]
    df_conditions = df_conditions[['PATIENT', 'DESCRIPTION']]

    df_master = pd.merge(df_patients, df_conditions, on='PATIENT', how='inner')

    # =================================================================
    # LA SOLUCIÓN PROFESIONAL: IMPUTACIÓN DE DATOS (Data Imputation)
    # En lugar de eliminar las filas con dropna(), rellenamos los valores nulos.
    # =================================================================
    df_master['ZIP'] = df_master['ZIP'].fillna('00000')
    df_master['RACE'] = df_master['RACE'].fillna('Unknown')
    df_master['MARITAL'] = df_master['MARITAL'].fillna('Unknown')

    df_master['Nombre_Real'] = df_master['FIRST'].astype(str) + ' ' + df_master['LAST'].astype(str)

    print(f"Base maestra consolidada: {len(df_master)} registros médicos mantenidos intactos.")

    # 4. Construcción del Escenario de Ataque con el nuevo arsenal
    qis = ['BIRTHDATE', 'GENDER', 'ZIP', 'RACE', 'MARITAL']

    public_registry = df_master[['Nombre_Real'] + qis].copy().drop_duplicates()
    hospital_data = df_master[qis + ['DESCRIPTION']].copy()
    hospital_data = hospital_data.sample(frac=1).reset_index(drop=True)

    # 5. LA METODOLOGÍA: Ejecutando el cruce dimensional
    print("Ejecutando algoritmo letal de re-identificación con imputación...")

    public_unicos = public_registry.drop_duplicates(subset=qis, keep=False)
    hospital_unicos = hospital_data.drop_duplicates(subset=qis, keep=False)

    ataque_exitoso = pd.merge(hospital_unicos, public_unicos, on=qis, how='inner')

    # 6. LOS RESULTADOS
    tasa_exito = (len(ataque_exitoso) / len(df_master)) * 100

    print("\n" + "="*65)
    print("🔬 RESULTADOS DEL EXPERIMENTO CORREGIDO (BIG DATA) - CIIS 2026")
    print("="*65)
    print(f"Población clínica total analizada:  {len(df_master)} registros.")
    print(f"Historiales médicos vulnerados:     {len(ataque_exitoso)}")
    print(f"Tasa de éxito de re-identificación: {tasa_exito:.2f}%")
    print("-" * 65)
    print("Primeros 5 registros expuestos (Notarás la etiqueta 'Unknown'):")
    print(ataque_exitoso[['Nombre_Real', 'DESCRIPTION', 'RACE', 'MARITAL', 'ZIP']].head())

In [ ]:
import os
import pandas as pd

print("Buscando las rutas exactas de los archivos masivos...")

ruta_pacientes = ""
ruta_condiciones = ""

for root, dirs, files in os.walk('.'):
    for file in files:
        if file.lower() == 'patients.csv':
            ruta_pacientes = os.path.join(root, file)
        elif file.lower() == 'conditions.csv':
            ruta_condiciones = os.path.join(root, file)

if not ruta_pacientes or not ruta_condiciones:
    print("❌ Error: No se encontraron los archivos CSV.")
else:
    print("✅ Archivos encontrados. Cargando bases de datos masivas...")
    df_patients = pd.read_csv(ruta_pacientes)
    df_conditions = pd.read_csv(ruta_condiciones)

    print("Procesando historiales clínicos con Imputación de Datos...")

    df_patients = df_patients.rename(columns={'Id': 'PATIENT'})
    df_patients = df_patients[['PATIENT', 'FIRST', 'LAST', 'BIRTHDATE', 'GENDER', 'ZIP', 'RACE', 'MARITAL']]
    df_conditions = df_conditions[['PATIENT', 'DESCRIPTION']]

    df_master = pd.merge(df_patients, df_conditions, on='PATIENT', how='inner')

    # Imputación de Datos (Manteniendo el millón de registros)
    df_master['ZIP'] = df_master['ZIP'].fillna('00000')
    df_master['RACE'] = df_master['RACE'].fillna('Unknown')
    df_master['MARITAL'] = df_master['MARITAL'].fillna('Unknown')

    df_master['Nombre_Real'] = df_master['FIRST'].astype(str) + ' ' + df_master['LAST'].astype(str)

    print(f"Base maestra consolidada: {len(df_master)} registros médicos mantenidos intactos.")

    # 4. Construcción del Escenario de Ataque
    qis = ['BIRTHDATE', 'GENDER', 'ZIP', 'RACE', 'MARITAL']

    # El padrón público SÍ tiene una fila por persona (eliminamos duplicados generados por el merge)
    public_registry = df_master[['Nombre_Real'] + qis].copy().drop_duplicates()

    # La base del hospital MANTIENE todas las enfermedades de los pacientes
    hospital_data = df_master[qis + ['DESCRIPTION']].copy()

    # 5. LA NUEVA METODOLOGÍA: Ataque Asimétrico
    print("Ejecutando algoritmo letal ASIMÉTRICO de re-identificación...")

    # PASO A: Encontramos a las personas que son ESTADÍSTICAMENTE ÚNICAS en el mundo real
    public_unicos = public_registry.drop_duplicates(subset=qis, keep=False)

    # PASO B: Cruzamos a estos individuos únicos con TODO el historial del hospital.
    # Ya no eliminamos duplicados en el hospital, así extraemos TODAS sus enfermedades.
    ataque_exitoso = pd.merge(hospital_data, public_unicos, on=qis, how='inner')

    # 6. LOS RESULTADOS
    tasa_exito = (len(ataque_exitoso) / len(df_master)) * 100

    print("\n" + "="*65)
    print("🔬 RESULTADOS DEL EXPERIMENTO FINAL (BIG DATA) - CIIS 2026")
    print("="*65)
    print(f"Población clínica total analizada:  {len(df_master)} registros.")
    print(f"Historiales médicos vulnerados:     {len(ataque_exitoso)}")
    print(f"Tasa de éxito de re-identificación: {tasa_exito:.2f}%")
    print("-" * 65)
    print("Primeros 5 registros expuestos (Notarás que una persona puede aparecer varias veces si tiene varias enfermedades):")
    print(ataque_exitoso[['Nombre_Real', 'DESCRIPTION', 'RACE', 'MARITAL', 'ZIP']].head(10))

In [ ]:
import os
import pandas as pd

print("Iniciando Ataque de Dimensionalidad Máxima...")

ruta_pacientes = ""
ruta_condiciones = ""

for root, dirs, files in os.walk('.'):
    for file in files:
        if file.lower() == 'patients.csv':
            ruta_pacientes = os.path.join(root, file)
        elif file.lower() == 'conditions.csv':
            ruta_condiciones = os.path.join(root, file)

if not ruta_pacientes or not ruta_condiciones:
    print("❌ Error: No se encontraron los archivos CSV.")
else:
    print("✅ Archivos masivos encontrados. Procesando...")
    df_patients = pd.read_csv(ruta_pacientes)
    df_conditions = pd.read_csv(ruta_condiciones)

    df_patients = df_patients.rename(columns={'Id': 'PATIENT'})

    # =================================================================
    # EL ARSENAL DEFINITIVO: Extraemos TODAS las dimensiones posibles
    # =================================================================
    qis = ['BIRTHDATE', 'GENDER', 'ZIP', 'RACE', 'MARITAL', 'ETHNICITY', 'CITY']

    columnas_pacientes = ['PATIENT', 'FIRST', 'LAST'] + qis
    df_patients = df_patients[columnas_pacientes]
    df_conditions = df_conditions[['PATIENT', 'DESCRIPTION']]

    df_master = pd.merge(df_patients, df_conditions, on='PATIENT', how='inner')

    # Imputación Masiva de Datos para no perder población
    for qi in qis:
        df_master[qi] = df_master[qi].fillna('Unknown')

    df_master['Nombre_Real'] = df_master['FIRST'].astype(str) + ' ' + df_master['LAST'].astype(str)

    print(f"Base maestra consolidada: {len(df_master)} registros médicos intactos.")

    # Construcción del Escenario de Ataque
    public_registry = df_master[['Nombre_Real'] + qis].copy().drop_duplicates()
    hospital_data = df_master[qis + ['DESCRIPTION']].copy()

    # Ejecutando el cruce letal
    print("Ejecutando cruce determinista al límite de entropía...")
    public_unicos = public_registry.drop_duplicates(subset=qis, keep=False)
    ataque_exitoso = pd.merge(hospital_data, public_unicos, on=qis, how='inner')

    # Resultados Definitivos
    tasa_exito = (len(ataque_exitoso) / len(df_master)) * 100

    print("\n" + "="*70)
    print("🔬 RESULTADOS DEL LÍMITE TEÓRICO (BIG DATA) - CIIS 2026")
    print("="*70)
    print(f"Población clínica total analizada:  {len(df_master)} registros.")
    print(f"Historiales médicos vulnerados:     {len(ataque_exitoso)}")
    print(f"TASA MÁXIMA DE RE-IDENTIFICACIÓN:   {tasa_exito:.2f}%")
    print("-" * 70)
    print("Primeros 5 registros del ataque de dimensionalidad máxima:")
    print(ataque_exitoso[['Nombre_Real', 'DESCRIPTION', 'CITY', 'RACE', 'MARITAL']].head())